In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

In [2]:
data = pd.read_csv("../DATASET/Plant_Parameters.csv")
target = data['Moisture']
features = data.drop('Moisture', axis=1)

le = LabelEncoder()
features['Plant Type'] = le.fit_transform(features['Plant Type'])

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2)

In [3]:
base_models = [
    ('gradient_boosting', GradientBoostingRegressor()),
    ('random_forest', RandomForestRegressor()),
    ('decision_tree', DecisionTreeRegressor())
]

meta_model = RandomForestRegressor()


In [4]:
def stacking_regression(base_models, meta_model, X_train, y_train, X_test, n_folds=5):
    kf = KFold(n_splits=n_folds)
    meta_features_train = np.zeros((X_train.shape[0], len(base_models)))
    meta_features_test = np.zeros((X_test.shape[0], len(base_models)))
    
    for i, (name, model) in enumerate(base_models):
        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]
            model.fit(X_train_fold, y_train_fold)
            meta_features_train[val_index, i] = model.predict(X_val_fold)
            meta_features_test[:, i] += model.predict(X_test)
        meta_features_test[:, i] /= n_folds
    
    meta_model.fit(meta_features_train, y_train)
    stacked_predictions = meta_model.predict(meta_features_test)
    
    return stacked_predictions

In [5]:
stacked_predictions = stacking_regression(base_models, meta_model, X_train, y_train, X_test)

mse = mean_squared_error(y_test, stacked_predictions)
r2 = r2_score(y_test, stacked_predictions)

print("Mean Squared Error:", mse)
print("R-squared:", r2)

# Optionally plot predictions vs. actual values
plt.figure()
plt.scatter(y_test, stacked_predictions)
plt.xlabel("Actual Moisture")
plt.ylabel("Predicted Moisture")
plt.title("Stacked Model Predictions")
plt.show()

KeyboardInterrupt: 